In [1]:
import pandas as pd
import logging
import json5

pd.options.display.max_columns = 400
pd.options.display.max_rows = 200

from modules import processRedCapFiles as rc
from modules import curationFunctions as cf
from modules import analysisFunctions as af

import ipywidgets as widgets


In [2]:
# floater_test = widgets.FloatSlider(
#     value=7.5,
#     min=0,
#     max=10.0,
#     step=0.1,
#     description='Test:',
#     disabled=False,
#     continuous_update=False,
#     orientation='horizontal',
#     readout=True,
#     readout_format='.1f',
# )

# display(floater_test)


In [3]:
# floater_test.value

In [4]:
logging.getLogger().setLevel(logging.INFO)

# <font color=teal>**TODO**</font>

Check for duplicate "Field Description" in a table

# <font color=yellow>**Configuration**</font>

In [5]:
class Configuration:
    def __init__(self, config_data):
        self.release = config_data.get("release", None)
        self.file_directory = config_data.get("file_directory", None)
        self.data_dictionary_path = config_data.get("data_dictionary_path", None)
        if "study_data" in config_data:
            self.study_id = config_data["study_data"].get("study_id", None)
            self.workspace_id = config_data["study_data"].get("workspace_id", None)
        else:
            self.study_id = None
            self.workspace_id = config_data["study_data"].get("workspace_id", None)

        self.all_data = config_data
    
    def get_tables_to_load(self):
        return self.all_data.get("tables_to_load")

    def get_table_metadata(self):
        return self.all_data.get("table_to_file")

    def get_table_metadata_df(self):
        data = self.get_table_metadata()
        return pd.DataFrame(data.items(),columns=["table_name","table_file"])


def load_configuration_file(filepath):
    f = open(filepath)
    config_data = Configuration(json5.load(f))
    
    return config_data


config_file = "../CoFar_SymLinks/SDY1515/curation_config.jsonc"

config_data = load_configuration_file(config_file)


In [6]:
release=config_data.release
study_id = config_data.study_id
workspace_id = config_data.workspace_id
directory = config_data.file_directory

# directory = "../CoFar_SymLinks/"+study_id+"/"
# data_dictionary = "CF4_Data_Dictionary_curated.csv"

# TODO: Move table_to_file into data dictionary
# table_to_file = "table_to_file.txt"

# data_dictionary_path = directory+"StudyFiles/"+data_dictionary
data_dictionary_path = config_data.data_dictionary_path

#This contains loaded data into tables.
table_metadata = config_data.get_table_metadata_df()


In [7]:
assessmentToPanel={}
study_tab_zip_file = f"{study_id}-DR{release}_Tab"
logging.basicConfig(filename=f"{study_id}.log", filemode='w', level=logging.DEBUG)


In [8]:
tables = config_data.get_tables_to_load()

In [9]:
type(tables)

list

# <font color=purple>**Setup**</font>
## Read study files
- [Data Dictionary]
- study_file.txt
- planned_visit.txt
## Read templates
- assessments

In [10]:
#Read study files
dictionary = rc.parseDataDictionary(data_dictionary_path)
study_files = cf.readFileFromZip(directory,study_tab_zip_file,"study_file.txt").drop(['STUDY_ACCESSION', 'WORKSPACE_ID'], axis=1)
studyfile_line_counts = cf.get_studyfile_line_count(f"../Data/{study_id}/StudyFiles")
study_files2 = study_files.merge(studyfile_line_counts, how='left', on='FILE_NAME')

planned_visits = cf.readFileFromZip(directory,study_tab_zip_file,"planned_visit.txt").drop(['STUDY_ACCESSION', 'WORKSPACE_ID'], axis=1)

#Read Templates
[assessment_panel_template,assessment_components_template,assessment_template_header] = cf.readTemplate("assessments")
assessment_components_template["ASSESSMENT_PANEL_ACCESSION"]=''


In [11]:
visit_info = planned_visits[["NAME","ORDER_NUMBER"]].sort_values(["ORDER_NUMBER"],ascending=([1]))


# <font color=green>Read eCRF Data files and load into assessment tables.</font>

In [12]:
tables[0]


{'tables': ['ADF'],
 'assessment_type': 'Skin Assessment',
 'template': 'assessments'}

In [14]:
[assessment_panel_template, assessment_components_template] = cf.processStudyFile(tables[0:13],directory,dictionary,planned_visits,study_files,assessment_panel_template,study_id,assessment_components_template,table_metadata,workspace_id)

assessment_components_template


Create new panel for files: cf4_adf_cofar4.txt
Create new panel for files: cf4_cmd_cofar4.txt
Create new panel for files: cf4_dm2_cofar4.txt
Create new panel for files: cf4_eet_cofar4.txt
Create new panel for files: cf4_ds4_cofar4.txt
Create new panel for files: cf4_es4_cofar4.txt
Create new panel for files: cf4_md1_cofar4.txt
Create new panel for files: cf4_md3_cofar4.txt
Create new panel for files: cf4_md4_cofar4.txt
Create new panel for files: cf4_ofc_cofar4.txt
Create new panel for files: cf4_spt_cofar4.txt


,User Defined ID,Planned Visit ID,Name Reported,Study Day,Age At Onset Reported,Age At Onset Unit Reported,Is Clinically Significant,Location Of Finding Reported,Organ Or Body System Reported,Result Value Reported,Result Unit Reported,Result Value Category,Subject Position Reported,Time Of Day,Verbatim Question,Who Is Assessed,ASSESSMENT_PANEL_ACCESSION,component_group_id,WORKSPACE_ID
0,SUB203766,PV7946,Comments,-9,NaN,NaN,NaN,NaN,NaN,reassessed for re challenge,NaN,NaN,NaN,NaN,Comments,,CCHMC_3,CCHMC_3_1,3133
1,SUB203768,PV7986,Comments,1093,NaN,NaN,NaN,NaN,NaN,Right toe eczema. Hydrocortisone cream applied.,NaN,NaN,NaN,NaN,Comments,,CCHMC_3,CCHMC_3_1,3133
2,SUB203768,PV7985,Comments,1164,NaN,NaN,NaN,NaN,NaN,Mild itch behind R knee.,NaN,NaN,NaN,NaN,Comments,,CCHMC_3,CCHMC_3_1,3133
3,SUB203774,PV10670,Comments,51,NaN,NaN,NaN,NaN,NaN,"Eczema over neck- flared. Per pt, usually flar...",NaN,NaN,NaN,NaN,Comments,,CCHMC_3,CCHMC_3_1,3133
4,SUB203780,PV10670,Comments,6,NaN,NaN,NaN,NaN,NaN,Does not have AD.,NaN,NaN,NaN,NaN,Comments,,CCHMC_3,CCHMC_3_1,3133
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
284124,SUB203790,PV7955,"PST, IgE, OFC since last",NaN,NaN,NaN,NaN,NaN,NaN,No,NaN,NaN,NaN,NaN,"Prick skin, serum IgE results and oral food ch...",,CCHMC_2,CCHMC_2_100,3133
284125,SUB203790,PV10670,"PST, IgE, OFC since last",NaN,NaN,NaN,NaN,NaN,NaN,No,NaN,NaN,NaN,NaN,"Prick skin, serum IgE results and oral food ch...",,CCHMC_2,CCHMC_2_100,3133
284126,SUB203791,PV7967,"PST, IgE, OFC since last",NaN,NaN,NaN,NaN,NaN,NaN,No,NaN,NaN,NaN,NaN,"Prick skin, serum IgE results and oral food ch...",,CCHMC_2,CCHMC_2_100,3133
284127,SUB203791,PV7985,"PST, IgE, OFC since last",NaN,NaN,NaN,NaN,NaN,NaN,No,NaN,NaN,NaN,NaN,"Prick skin, serum IgE results and oral food ch...",,CCHMC_2,CCHMC_2_100,3133


In [33]:
len(assessment_components_template[~assessment_components_template["Result Value Reported"].isnull()])

278303

# <font color=red>**Run Quality Checks**</font>
 - Missing Visits
 - Missing Columns

In [15]:
### Missing Visits
cf.missingVisits_all

{}

# <font color=Violet>**Write to template files**</font>

In [16]:
# TODO Need to add in top 2 rows from template that contain version, and "Please dont delete column" text
# output_filename = "curations/assessments_test2.tsv"
output_filename = "curations/assessments_%s.tsv"%study_id
cf.writePanelComponentTemplate(assessment_panel_template, assessment_components_template,assessment_template_header, output_filename)
output_filename



'curations/assessments_SDY1515.tsv'

In [ ]:
"".join(assessment_template_header.columns)


# <font color=red>**Playground**</font>
#### This area for writing code to query/find/analyze 

In [ ]:
# af.showRelevantColumns(assessment_components_template[(assessment_components_template["Name Reported"] == "PST shellfish quan result") & (~assessment_components_template["Result Value Reported"].isna())])
thisdata = af.showRelevantColumns(assessment_components_template[(assessment_components_template["Name Reported"].isin(
    [
"IgE egg greater_less",
"IgE egg performed",
"IgE egg qual result",
"IgE egg quan result",
"IgE egg test location"
])
) & (assessment_components_template["User Defined ID"] == "SUB203788")])

thisdata
thisdata[["User Defined ID","Planned Visit ID","Name Reported","Study Day","Result Value Reported","Result Unit Reported","Verbatim Question"]]

In [ ]:
# af.showRelevantColumns(assessment_components_template[(assessment_components_template["Name Reported"] == "PST shellfish quan result") & (~assessment_components_template["Result Value Reported"].isna())])
thisdata = af.showRelevantColumns(assessment_components_template[(assessment_components_template["Name Reported"].isin(
    ["OFC egg performed",
"OFC egg test location",
"OFC egg test result",
"OFC fish chall method"])
) & (assessment_components_template["User Defined ID"] == "SUB203763")])

thisdata
thisdata[["User Defined ID","Planned Visit ID","Name Reported","Study Day","Result Value Reported","Verbatim Question"]]

In [ ]:
# af.showRelevantColumns(assessment_components_template[(assessment_components_template["Name Reported"] == "PST shellfish quan result") & (~assessment_components_template["Result Value Reported"].isna())])
thisdata = af.showRelevantColumns(assessment_components_template[(assessment_components_template["Name Reported"].isin(
    [

"Egg lower resp reaction",
"Egg CV reaction",
"Egg GI reaction",
"Egg oral reaction",
"Egg skin reaction",
"Egg upper resp reaction",
"Egg symptoms date"


])
)])# & (assessment_components_template["User Defined ID"] == "SUB203763")])

thisdata
thisdata[["User Defined ID","Planned Visit ID","Name Reported","Study Day","Result Value Reported","Verbatim Question"]]

In [ ]:

# ages = ad[ad["ASSESSMENT_PANEL_ACCESSION"]==ap][["SUBJECT_ACCESSION","STUDY_DAY","VERBATIM_QUESTION","NAME_REPORTED_y","RESULT_VALUE_REPORTED","AGE_AT_ONSET_REPORTED","AGE_AT_ONSET_UNIT_REPORTED"]].groupby(["SUBJECT_ACCESSION","STUDY_DAY","AGE_AT_ONSET_REPORTED","AGE_AT_ONSET_UNIT_REPORTED"], dropna=False).size().reset_index(name='Count').sort_values(by=["SUBJECT_ACCESSION","STUDY_DAY"])

assessment_components_template[["Planned Visit ID","Study Day","ASSESSMENT_PANEL_ACCESSION"]].groupby(["Planned Visit ID","Study Day","ASSESSMENT_PANEL_ACCESSION"], dropna=False).size().reset_index(name='Count').head(50)


In [ ]:
df_temp

In [ ]:
#def writeTemplatefile
assessment_panel_template


In [ ]:
assessment_components_template

In [ ]:
af.showRelevantColumns(assessment_components_template[assessment_components_template["Planned Visit ID"] == ""])


In [ ]:
af.showRelevantColumns(assessment_panel_template)


In [ ]:
assessment_panel_template
#=cf.datafileToComponents(datafile,dictionary,file_table,panel_id,assessment_components_template,workspace_id)
#assessment_components_template

In [ ]:
# col_mappings = assessment_file_metadata[file_table]["col_mappings"]
# col_units = assessment_file_metadata[file_table]["col_units"]
# start_pivot_column = cf.getColumnNumber(datafile,assessment_file_metadata[file_table]["start_pivot_column_name"])

# # assessment_components_template=cf.datafileToComponents(datafile,start_pivot_column,panel_id,assessment_components_template,workspace_id,col_mappings,col_units)
# assessment_components_template=cf.datafileToComponents(datafile,dictionary,file_table,panel_id,assessment_components_template,workspace_id,)

# assessment_components_template.head(2)


In [ ]:
assessment_components_template[assessment_components_template["Name Reported"]=="Full sib 1 allergic rhini"].head()

In [ ]:
component_questions = assessment_components_template.groupby(["component_group_id","ASSESSMENT_PANEL_ACCESSION","Name Reported"], dropna=False).size().reset_index(name='Count')

component_questions

In [ ]:
# datafile.groupby(["Visit Number","PLANNED_VISIT_ID"], as_index=False).agg(
#     count=("Accession","nunique")
# ).sort_values(["Visit Number"],ascending=([1]))